# 04b — t-SNE Explained

## The One-Sentence Version
t-SNE preserves who-is-near-whom: if two points are neighbours in high-D, they should be neighbours in the 2D map.

## What You'll Build Intuition For
- What t-SNE optimises (matching neighbourhood similarity distributions)
- How perplexity controls local vs global focus
- What t-SNE does well (local cluster structure) and what it distorts
- Practical pitfalls that trip up beginners

## Prerequisites
- [04a — Why Linear Fails](04a_why_linear_fails.ipynb) — why we need nonlinear methods

## The Story

Imagine you're arranging photos on a table. You have thousands of photos and you want similar ones near each other — cats near cats, dogs near dogs. You can't preserve *all* the distances (the table is 2D, the photos differ in thousands of ways), so you focus on what matters most: **nearby things should stay nearby.**

That's exactly what t-SNE does. It asks: "In high-D, which points are neighbours?" Then it arranges points in 2D so that those same pairs end up close together. It doesn't care about preserving distances between distant points — only the local neighbourhoods.

This gives t-SNE its distinctive look: tight, well-separated clusters where local structure is beautifully preserved. But it also means you can't trust the distances *between* clusters, or the sizes of the clusters. Those are artefacts of the optimisation.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data, make_swiss_roll_data

apply_style()
rng = np.random.default_rng(42)

## The Core Idea

t-SNE works in two steps:

1. **In high-D:** For each point, compute a probability distribution over all other points — nearby points get high probability, far points get low probability (Gaussian kernel).

2. **In low-D:** Place points in 2D and compute the same kind of probabilities, but using a Student-t kernel (heavier tails — this is the "t" in t-SNE).

3. **Optimise:** Move the 2D points until the low-D probabilities match the high-D probabilities as closely as possible (minimise KL divergence).

The heavy-tailed Student-t kernel in the low-D space is the key trick: it allows moderately distant high-D points to be pushed far apart in 2D, creating the clean separation between clusters.

In [ ]:
# t-SNE on the digits dataset
data, target, images = make_digit_data()

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(data)

fig, ax = plt.subplots(figsize=(10, 8))
for digit in range(10):
    mask = target == digit
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=12, alpha=0.6,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("t-SNE on digits: beautiful cluster separation")
ax.legend(title="Digit", markerscale=3, fontsize=9)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.savefig("visuals/04b_tsne_digits.png", dpi=150, bbox_inches="tight")
plt.show()

print("Each digit class forms a tight cluster.")
print("Compare this to PCA's blurry overlap — t-SNE is dramatically better")
print("at revealing local cluster structure.")

## Perplexity: The One Knob That Matters

Perplexity controls how many effective neighbours each point "pays attention to."

- **Low perplexity (5):** Very local — each point only cares about its 5 nearest neighbours. Many small, tight clusters.
- **High perplexity (50-100):** Broader view — each point considers more neighbours. Fewer, larger clusters.

Typical range: 5–50. The default (30) is a reasonable starting point.

In [ ]:
# Same data, different perplexity values
perplexities = [5, 15, 30, 50, 100]

fig, axes = plt.subplots(1, len(perplexities), figsize=(18, 4))

for ax, perp in zip(axes, perplexities):
    tsne_p = TSNE(n_components=2, perplexity=perp, random_state=42)
    X_p = tsne_p.fit_transform(data)

    for digit in range(10):
        mask = target == digit
        ax.scatter(X_p[mask, 0], X_p[mask, 1], s=5, alpha=0.5,
                   color=PALETTE[digit % len(PALETTE)])
    ax.set_title(f"perplexity = {perp}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Perplexity: from hyper-local (5) to broad (100)", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/04b_perplexity_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

print("Low perplexity → many small sub-clusters (possibly fragmenting real groups).")
print("High perplexity → fewer clusters, broader structure.")
print("Always try multiple values.")

## What t-SNE Gets Right

t-SNE excels at **local structure preservation**. Points that are genuinely similar in high-D end up close in 2D. This makes it superb for:
- Finding clusters and sub-clusters
- Visualising class separation
- Exploring high-dimensional data for the first time

In [ ]:
# t-SNE on Swiss roll — handles the curvature PCA couldn't
X_swiss, colour_swiss = make_swiss_roll_data(n=1500, noise=0.3, seed=42)

pca_swiss = PCA(n_components=2).fit_transform(X_swiss)
tsne_swiss = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_swiss)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(pca_swiss[:, 0], pca_swiss[:, 1], s=5, alpha=0.5,
            c=colour_swiss, cmap="viridis")
ax1.set_title("PCA: colours overlap")
ax1.set_xticks([])
ax1.set_yticks([])

ax2.scatter(tsne_swiss[:, 0], tsne_swiss[:, 1], s=5, alpha=0.5,
            c=colour_swiss, cmap="viridis")
ax2.set_title("t-SNE: colour gradient preserved")
ax2.set_xticks([])
ax2.set_yticks([])

fig.suptitle("Swiss Roll: PCA vs t-SNE", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/04b_tsne_swiss.png", dpi=150, bbox_inches="tight")
plt.show()

print("t-SNE keeps nearby colours together — it respects the manifold structure.")

## What t-SNE Gets Wrong

t-SNE has well-known failure modes. Understanding them prevents misinterpretation.

1. **Cluster sizes are meaningless** — dense regions get expanded, sparse regions get compressed
2. **Distances between clusters are meaningless** — the inter-cluster gaps are artefacts
3. **Different runs give different layouts** — without a fixed seed, the arrangement changes
4. **It's slow** — O(n²) by default, O(n log n) with Barnes-Hut approximation

In [ ]:
# Different random seeds → different layouts, same clusters
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, seed in zip(axes, [42, 123, 999]):
    tsne_run = TSNE(n_components=2, perplexity=30, random_state=seed)
    X_run = tsne_run.fit_transform(data)

    for digit in range(10):
        mask = target == digit
        ax.scatter(X_run[mask, 0], X_run[mask, 1], s=8, alpha=0.5,
                   color=PALETTE[digit % len(PALETTE)])
    ax.set_title(f"seed = {seed}")
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle("Three t-SNE runs: same clusters, different arrangement", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/04b_tsne_instability.png", dpi=150, bbox_inches="tight")
plt.show()

print("The clusters are consistent — the same digits always group together.")
print("But the spatial arrangement changes. Don't read meaning into cluster positions.")

## Practical Pitfalls

| Pitfall | Why It's Dangerous |
|---------|-------------------|
| Using t-SNE output as features for ML | The coordinates are optimisation artefacts, not meaningful features |
| Interpreting cluster sizes | Dense high-D regions get inflated; sparse ones shrink |
| Interpreting inter-cluster distances | They depend on perplexity and initialisation, not real distances |
| Running only one perplexity | Different values reveal different structure levels |
| Using t-SNE on tiny datasets | Fewer than ~50 points? t-SNE has nothing to work with |

> **Rule of thumb:** t-SNE is a **visualisation tool**, not a preprocessing step. Look at it, learn from it, but never feed its output into another algorithm.

<details>
<summary><b>The Maths Behind This</b></summary>

**High-D similarities:** For each pair $(i, j)$, compute a conditional probability:

$$p_{j|i} = \frac{\exp(-\|\mathbf{x}_i - \mathbf{x}_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|\mathbf{x}_i - \mathbf{x}_k\|^2 / 2\sigma_i^2)}$$

The bandwidth $\sigma_i$ is set so that the perplexity (effective number of neighbours) matches the user's choice. Symmetrise: $p_{ij} = (p_{j|i} + p_{i|j}) / 2n$.

**Low-D similarities:** Using a Student-t kernel with 1 degree of freedom:

$$q_{ij} = \frac{(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|\mathbf{y}_k - \mathbf{y}_l\|^2)^{-1}}$$

**Loss:** KL divergence $\text{KL}(P \| Q) = \sum_{ij} p_{ij} \log \frac{p_{ij}}{q_{ij}}$

The asymmetric KL divergence means: if $p_{ij}$ is large (points are neighbours in high-D) but $q_{ij}$ is small (they're far in low-D), the cost is high. But if $p_{ij}$ is small and $q_{ij}$ is large, the cost is low. t-SNE prioritises keeping neighbours together over pushing non-neighbours apart.

</details>

## Where This Matters

**Single-cell RNA-seq:** The most common application. Biologists use t-SNE (and now UMAP) to visualise thousands of cells, each described by ~20,000 gene expression values. Clusters in the t-SNE plot correspond to cell types.

**Anomaly detection exploration:** Fraud analysts use t-SNE to spot unusual transactions. Normal transactions cluster together; outliers appear as isolated points or small groups. It's a visual first pass before building a formal model.

## Feynman Check

1. **What is t-SNE trying to preserve?**

2. **Why can't you trust the distance between two t-SNE clusters?**

3. **What does perplexity control, in plain language?**

t-SNE showed us the power of nonlinear embedding. In **04c — UMAP Explained**, we'll see a faster, often better alternative.